# Day 01: Python for ML — NumPy, Matrix Ops, Vectorization

**Goal:** Get comfortable with the math operations that power every LLM.

Everything in an LLM is matrix math. A word is a vector, a sentence is a matrix, and a neural network layer is just `output = input @ weights`. Let's build that intuition.

## 1. NumPy Arrays vs Python Lists

Python lists are slow — they loop element by element. NumPy arrays operate on the entire array at once (vectorized), making them **100x faster**.

In [ ]:
import numpy as np
import time

# Python list way — slow
a_list = list(range(1_000_000))
b_list = list(range(1_000_000))

start = time.time()
result_list = [a_list[i] + b_list[i] for i in range(len(a_list))]
list_time = time.time() - start

# NumPy way — fast
a_np = np.arange(1_000_000)
b_np = np.arange(1_000_000)

start = time.time()
result_np = a_np + b_np
np_time = time.time() - start

print(f"Python list: {list_time:.4f}s")
print(f"NumPy array: {np_time:.4f}s")
print(f"NumPy is {list_time / np_time:.0f}x faster!")

## 2. Shapes and Dimensions

Shape is the #1 thing you'll debug in ML. Every tensor/array has a shape, and operations require compatible shapes.

```
Scalar:  42                  → shape: ()      — just a number
Vector:  [1, 2, 3]          → shape: (3,)    — like a word embedding
Matrix:  [[1, 2], [3, 4]]   → shape: (2, 2)  — like a weight matrix
3D:      batch of matrices   → shape: (B, R, C) — batch of sentences
```

In [ ]:
# Creating arrays of different dimensions
scalar = np.array(42)
vector = np.array([1, 2, 3])
matrix = np.array([[1, 2, 3], [4, 5, 6]])
tensor_3d = np.array([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])

print(f"Scalar:    value={scalar}, shape={scalar.shape}, ndim={scalar.ndim}")
print(f"Vector:    value={vector}, shape={vector.shape}, ndim={vector.ndim}")
print(f"Matrix:    shape={matrix.shape}, ndim={matrix.ndim}")
print(f"3D Tensor: shape={tensor_3d.shape}, ndim={tensor_3d.ndim}")

print(f"\nMatrix contents:\n{matrix}")

In [ ]:
# Reshaping — same data, different shape
# This is used ALL the time in ML to prepare data for layers

a = np.arange(12)  # [0, 1, 2, ..., 11]
print(f"Original: shape={a.shape} → {a}")

b = a.reshape(3, 4)  # 3 rows, 4 columns
print(f"\nReshaped to (3,4):\n{b}")

c = a.reshape(2, 2, 3)  # 2 batches of 2x3 matrices
print(f"\nReshaped to (2,2,3):\n{c}")

# -1 means "figure it out automatically"
d = a.reshape(4, -1)  # 4 rows, numpy figures out columns = 3
print(f"\nReshaped to (4,-1) → shape {d.shape}:\n{d}")

## 3. Dot Product — "How Similar Are Two Vectors?"

The dot product multiplies element-wise and sums up. It tells you how much two vectors point in the same direction.

In LLMs, this is how **attention** works — the dot product between two word vectors tells the model how much one word should "attend to" another.

```
a = [1, 2, 3]
b = [4, 5, 6]
dot = 1×4 + 2×5 + 3×6 = 32
```

In [ ]:
# Dot product — by hand vs NumPy
a = np.array([1, 2, 3])
b = np.array([4, 5, 6])

# Manual way
dot_manual = sum(a[i] * b[i] for i in range(len(a)))

# NumPy way
dot_numpy = np.dot(a, b)

# Also works with @ operator
dot_at = a @ b

print(f"Manual:  {dot_manual}")
print(f"np.dot:  {dot_numpy}")
print(f"a @ b:   {dot_at}")

# Similarity example: similar vectors have high dot product
cat = np.array([0.9, 0.1, 0.8])    # imagine: [furry, wheels, small]
dog = np.array([0.8, 0.1, 0.6])    # similar to cat
car = np.array([0.0, 0.9, 0.2])    # very different

print(f"\ncat · dog = {cat @ dog:.2f}  (similar — both animals)")
print(f"cat · car = {cat @ car:.2f}  (different — animal vs vehicle)")

## 4. Matrix Multiplication — The Core of Neural Networks

Every neural network layer does: `output = input @ weights`

**Rule:** For `A @ B`, the inner dimensions must match.
- A has shape (m, **n**) and B has shape (**n**, p) → result is (m, p)
- Think: (2, **3**) @ (**3**, 4) = (2, 4) ✓
- (2, **3**) @ (**4**, 2) = ERROR ✗

In [ ]:
# Matrix multiplication
A = np.array([[1, 2, 3],
              [4, 5, 6]])  # shape (2, 3)

B = np.array([[7, 10],
              [8, 11],
              [9, 12]])    # shape (3, 2)

C = A @ B  # (2, 3) @ (3, 2) = (2, 2)
print(f"A shape: {A.shape}")
print(f"B shape: {B.shape}")
print(f"C = A @ B shape: {C.shape}")
print(f"C:\n{C}")

# Let's verify one element by hand:
# C[0,0] = A[0,:] · B[:,0] = 1*7 + 2*8 + 3*9 = 7 + 16 + 27 = 50
print(f"\nVerify C[0,0]: 1×7 + 2×8 + 3×9 = {1*7 + 2*8 + 3*9}")

In [ ]:
# Simulate a neural network layer!
# This is EXACTLY what happens inside a real model

# 3 input features (e.g., 3-dimensional word embedding)
input_data = np.array([0.5, 0.8, 0.2])  # shape (3,)

# Weight matrix: transforms 3 features → 4 features
weights = np.random.randn(3, 4)  # shape (3, 4)

# Bias: one per output feature
bias = np.random.randn(4)  # shape (4,)

# Forward pass: output = input @ weights + bias
output = input_data @ weights + bias

print(f"Input:   shape {input_data.shape} → {input_data}")
print(f"Weights: shape {weights.shape}")
print(f"Bias:    shape {bias.shape}")
print(f"Output:  shape {output.shape} → {output}")
print(f"\nThis is one layer of a neural network. Stack many of these → LLM!")

## 5. Broadcasting — Auto-expanding Shapes

NumPy automatically stretches smaller arrays to match larger ones. This avoids writing loops and is used everywhere in ML (e.g., adding a bias to every row).

In [ ]:
# Broadcasting examples

# Matrix (3, 3) + Vector (3,) → vector is added to EACH row
matrix = np.array([[1, 2, 3],
                    [4, 5, 6],
                    [7, 8, 9]])
bias = np.array([10, 20, 30])

result = matrix + bias
print(f"Matrix (3,3) + Bias (3,):")
print(result)

# This is exactly what "add bias" does in a neural network:
# each sample in the batch gets the same bias added

# Scalar broadcast — multiply everything by 2
print(f"\nMatrix * 2:\n{matrix * 2}")

# Column broadcast — need to reshape to (3, 1)
column = np.array([[100], [200], [300]])  # shape (3, 1)
print(f"\nMatrix + Column (3,1):\n{matrix + column}")

## 6. Useful NumPy Operations for ML

A quick reference of operations you'll use constantly.

In [ ]:
data = np.array([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])

# Sum, mean, max — with axis control
print(f"Sum of all:     {data.sum()}")
print(f"Sum per column: {data.sum(axis=0)}")  # collapse rows → (3,)
print(f"Sum per row:    {data.sum(axis=1)}")  # collapse cols → (2,)
print(f"Mean:           {data.mean()}")
print(f"Max per row:    {data.max(axis=1)}")
print(f"Argmax per row: {data.argmax(axis=1)}")  # index of max — used in classification

# Transpose — swap rows and columns
print(f"\nOriginal shape: {data.shape}")
print(f"Transposed shape: {data.T.shape}")

# Random — used to initialize weights
random_weights = np.random.randn(3, 4)  # normal distribution, mean=0, std=1
print(f"\nRandom weights (3,4):\n{random_weights}")

# Zeros and ones — used for initialization
print(f"\nZeros: {np.zeros(5)}")
print(f"Ones:  {np.ones(5)}")

## 7. Mini Project: Fake Word Similarity Engine

Let's put it all together. We'll create fake word embeddings and build a similarity search — this is a tiny version of what powers RAG and search in LLM apps.

In [ ]:
# Mini Project: Word Similarity using Dot Products
# In real LLMs, each word is a vector of 768-4096 dimensions
# We'll use 5 dimensions to keep it visible

# Fake word embeddings (imagine these were learned by a model)
words = {
    "king":    np.array([0.9, 0.1, 0.8, 0.2, 0.7]),
    "queen":   np.array([0.8, 0.9, 0.7, 0.2, 0.6]),
    "man":     np.array([0.8, 0.1, 0.3, 0.5, 0.4]),
    "woman":   np.array([0.7, 0.9, 0.3, 0.5, 0.3]),
    "dog":     np.array([0.2, 0.3, 0.1, 0.9, 0.8]),
    "cat":     np.array([0.2, 0.4, 0.1, 0.8, 0.7]),
    "car":     np.array([0.1, 0.1, 0.9, 0.1, 0.2]),
    "bicycle": np.array([0.1, 0.1, 0.7, 0.2, 0.3]),
}

def cosine_similarity(a, b):
    """Dot product normalized by magnitudes — gives -1 to 1 similarity."""
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))

def find_similar(query_word, top_n=3):
    """Find the most similar words to query_word."""
    query = words[query_word]
    similarities = {}
    for word, vec in words.items():
        if word != query_word:
            similarities[word] = cosine_similarity(query, vec)
    
    # Sort by similarity (highest first)
    ranked = sorted(similarities.items(), key=lambda x: x[1], reverse=True)
    return ranked[:top_n]

# Try it out!
for query in ["king", "dog", "car"]:
    print(f"\nMost similar to '{query}':")
    for word, score in find_similar(query):
        print(f"  {word:10s} → {score:.3f}")

In [ ]:
# The famous Word2Vec analogy: king - man + woman ≈ queen
result = words["king"] - words["man"] + words["woman"]

print("king - man + woman = ?")
print(f"Result vector: {result}\n")

# Find which word is closest to this result
for word, vec in words.items():
    sim = cosine_similarity(result, vec)
    marker = " ← closest!" if word == "queen" else ""
    print(f"  {word:10s} → {sim:.3f}{marker}")

## Exercises (try these yourself!)

1. **Shape detective:** Create a (4, 3) matrix and a (3, 5) matrix. What shape is the result of `A @ B`? Verify with code.

2. **Vectorization speed test:** Compare the speed of computing `sum(x**2)` using a Python loop vs `np.sum(x**2)` for an array of 10 million elements.

3. **Add more words** to the similarity engine above. Try adding "prince", "princess", "truck". Do the similarities make sense?

4. **Matrix chain:** If you have 3 matrices A(2,3), B(3,4), C(4,1) — what's the shape of `A @ B @ C`? What does shape (2,1) mean in terms of neural networks? (hint: 2 inputs → 1 output)

---

## Key Takeaways

- Everything in ML is matrix/vector math
- **Shape** is the #1 thing to track — most bugs are shape mismatches
- **Dot product** measures similarity — this powers attention in transformers
- **Matrix multiplication** = one layer of a neural network: `output = input @ weights + bias`
- **Vectorization** = no loops, let NumPy handle it in C
- Tomorrow: how does the model learn? → gradients and calculus intuition